# Autoresearch 실험 분석 — pink IK pick-place SM

`results.tsv`(자율 실험 로그) 시각화. 메트릭 = **success_rate**(그릇 안착률, 높을수록 좋음),
보조 = **ever_rate**(한 번이라도 그릇 안 = pick 은 성공).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 탭 구분 5열: commit, success_rate, ever_rate, status, description
df = pd.read_csv("results.tsv", sep="\t")
df["success_rate"] = pd.to_numeric(df["success_rate"], errors="coerce")
df["ever_rate"] = pd.to_numeric(df["ever_rate"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"실험 수: {len(df)}")
counts = df["status"].value_counts()
print(counts.to_string())
n_keep, n_disc = counts.get("KEEP", 0), counts.get("DISCARD", 0)
if n_keep + n_disc:
    print(f"keep rate: {n_keep}/{n_keep + n_disc} = {n_keep / (n_keep + n_disc):.1%}")
df.tail(10)

In [ ]:
# keep 된 실험(개선이 살아남은 것) 일람
kept = df[df["status"] == "KEEP"]
for i, row in kept.iterrows():
    print(f"  #{i:3d}  success={row['success_rate']:.2f}  ever={row['ever_rate']:.2f}  {row['description']}")

## 진행 그래프

실험 순서에 따른 success_rate. 초록 = keep, 회색 = discard, 계단선 = 지금까지의 최고 기록.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
valid = df[df["status"] != "CRASH"].reset_index(drop=True)

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["success_rate"], c="#cccccc", s=14, alpha=0.6, zorder=2, label="discard")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["success_rate"], c="#2ecc71", s=55, zorder=4,
           edgecolors="black", linewidths=0.5, label="keep")
ax.scatter(kept_v.index, kept_v["ever_rate"], marker="^", c="#3498db", s=30,
           alpha=0.6, zorder=3, label="ever (pick만)")

running_best = kept_v["success_rate"].cummax()
ax.step(kept_v.index, running_best, where="post", color="#27ae60", linewidth=2, alpha=0.7,
        zorder=3, label="running best")

for idx, row in kept_v.iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, row["success_rate"]), textcoords="offset points",
                xytext=(6, 6), fontsize=8, color="#1a7a3a", rotation=30, ha="left", va="bottom")

ax.axhline(1.0, color="#e74c3c", linestyle="--", alpha=0.5, linewidth=1, label="목표 1.0")
ax.set_xlabel("실험 #")
ax.set_ylabel("success_rate (그릇 안착률)")
ax.set_ylim(-0.05, 1.1)
ax.set_title(f"Autoresearch 진행: {len(df)} 실험, {len(kept_v)} keep")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved: progress.png")

## 요약

In [ ]:
kept = df[df["status"] == "KEEP"]
base = df.iloc[0]
best = kept.loc[kept["success_rate"].idxmax()]
print(f"baseline : success={base['success_rate']:.2f} ever={base['ever_rate']:.2f}")
print(f"best     : success={best['success_rate']:.2f} ever={best['ever_rate']:.2f}")
print(f"개선     : +{best['success_rate'] - base['success_rate']:.2f}")
print(f"best 실험: {best['description']}")
print()
# keep 별 개선폭(직전 keep 대비) — 뭐가 효자였나
k = kept.copy()
k["delta"] = k["success_rate"].diff()
hits = k.iloc[1:].sort_values("delta", ascending=False)
print("keep 별 개선폭:")
for _, row in hits.iterrows():
    print(f"  {row['delta']:+.2f}  → {row['success_rate']:.2f}  {row['description']}")